<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/14-structured-streaming/06-checkpointing-semantics.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 14.6 — Checkpointing: what's actually in there, and restart behavior

List the checkpoint directory's actual contents (offsets, commits,
state), then prove restart-without-reprocessing using the file source
from 14.1 — stop a query, add more files, restart with the SAME
checkpoint dir, and confirm old files are not reprocessed.

In [ ]:
import os, sys, shutil, time, json
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("14.6-checkpointing-semantics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## What's inside a checkpoint directory

In [ ]:
shutil.rmtree("stream_in", ignore_errors=True)
shutil.rmtree("checkpoints/ckpt_demo", ignore_errors=True)
os.makedirs("stream_in", exist_ok=True)

from pyspark.sql.types import StructType, StructField, StringType, DoubleType
schema = StructType([StructField("customer_id", StringType()), StructField("amount", DoubleType())])

def emit_file(name, rows):
    with open(f"stream_in/{name}", "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

for i in range(2):
    emit_file(f"f{i}.json", [{"customer_id": f"c{i}", "amount": float(i)}])

q1 = (spark.readStream.schema(schema).json("stream_in/")
      .writeStream.format("memory").queryName("ckpt_demo")
      .option("checkpointLocation", "checkpoints/ckpt_demo")
      .outputMode("append").start())

time.sleep(3)
print("rows processed so far:", spark.sql("SELECT count(*) AS n FROM ckpt_demo").collect()[0]["n"])
q1.stop()

print("\ncheckpoint directory contents:")
for root, dirs, files in os.walk("checkpoints/ckpt_demo"):
    level = root.replace("checkpoints/ckpt_demo", "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for f in files:
        print("  " * (level + 1) + f)

## Restart with the SAME checkpoint: old files are NOT reprocessed

Add two more files, restart against the same checkpoint dir, confirm
the memory table only gains the NEW rows -- proof the offsets in the
checkpoint were actually honored.

In [ ]:
for i in range(2, 4):
    emit_file(f"f{i}.json", [{"customer_id": f"c{i}", "amount": float(i)}])

q2 = (spark.readStream.schema(schema).json("stream_in/")
      .writeStream.format("memory").queryName("ckpt_demo")   # SAME query name
      .option("checkpointLocation", "checkpoints/ckpt_demo")  # SAME checkpoint dir
      .outputMode("append").start())

time.sleep(3)
print("rows after restart + 2 new files (should be 4 total, not 4+2 reprocessed):")
spark.sql("SELECT * FROM ckpt_demo ORDER BY customer_id").show()
q2.stop()

## Idempotent `foreachBatch` — the sink-side half of exactly-once

Spark's offset tracking is precise; the SINK must independently avoid
double-applying a re-run batch. This dedup-by-key pattern is the
minimum idempotency contract 14.6 asks for.

In [ ]:
written_keys = set()   # stand-in for "a real idempotent sink's unique constraint"

def idempotent_write(batch_df, batch_id):
    rows = batch_df.collect()
    new = [r for r in rows if r["customer_id"] not in written_keys]
    for r in new:
        written_keys.add(r["customer_id"])
    print(f"batch {batch_id}: {len(rows)} rows seen, {len(new)} newly applied "
          f"(dedup prevents double-apply on a re-run of this same batch)")

shutil.rmtree("checkpoints/idem_demo", ignore_errors=True)
idem_q = (spark.readStream.schema(schema).json("stream_in/")
          .writeStream.foreachBatch(idempotent_write)
          .option("checkpointLocation", "checkpoints/idem_demo").start())
time.sleep(3)
idem_q.stop()
print("final applied set:", written_keys)

In [ ]:
shutil.rmtree("stream_in", ignore_errors=True)
shutil.rmtree("checkpoints", ignore_errors=True)
spark.stop()

## Exercises

1. After the first run (`q1`), manually delete the
   `checkpoints/ckpt_demo/offsets` subdirectory only (leave the rest)
   and restart. What happens — does it reprocess everything, nothing,
   or error? What does this tell you about which piece of the
   checkpoint actually tracks "what have I read"?
2. Run the restart demo again, but this time start the SECOND query
   with a DIFFERENT `queryName` and a DIFFERENT checkpoint directory.
   Does it reprocess the original 2 files? Why does query identity /
   checkpoint path matter here?
3. In the idempotent `foreachBatch` demo, simulate a "re-run" by
   calling `idempotent_write(batch_df, 0)` a second time manually with
   the same `batch_id`. Confirm `written_keys` doesn't grow — that's
   the idempotency contract working.
4. Time how long the checkpoint directory listing takes to grow (in
   number of files) as you emit more file-source batches. What does
   this suggest about checkpoint directories needing their own
   lifecycle management (compaction, cleanup) on a long-running
   production query?
5. Explain, using what you observed, why "delete the checkpoint to
   unstick a query" is dangerous — reproduce the scenario by deleting
   the WHOLE checkpoint dir after `q1` and restarting fresh. What
   happens to the 2 already-processed files?

In [ ]:
# your exercise code here